In [1]:

# ================================
# Airbnb Listing Analysis
# Christopher Newman
# ================================

# Core Libraries
import pandas as pd # Data manipulation and analysis, helps load, clean, filtter, and organize the dataset as a DF (DataFrame)
import numpy as np #Numerical Operations - helps with numerical calculation, missing values, and array-based operations

# Visualization
import matplotlib.pyplot as plt #Basic data visualization - is used to create charts 
import seaborn as sns # Stats data viz - Makes it easier to create cleaner looking plots, especially for distributions and correlations
import plotly.express as px # Interactive viz - Express can be used for interactive charts

# Machine Learning
from sklearn.model_selection import train_test_split # This lets the model learn from one portion of the data and be evaluated on unseen data
from sklearn.linear_model import LinearRegression # Linear Regression is the baseline model used to PREDICT Airbnb price from listing
from sklearn.metrics import mean_squared_error, r2_score # RMSE shows the avg prediction error in dollars, R-squared shows how much the price varitation the model explains
from sklearn.preprocessing import OneHotEncoder # OneHotEncoder converts text categories like room_type or city into numeric colmuns so the model can understand
from sklearn.ensemble import RandomForestRegressor

# Combining preprocessing and modeling steps
# ColmumnTransformer and Pipeline help keep cleaning, encoding, and modleing organized in one workflow.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Utilities
from datetime import datetime
import re


In [2]:
# Load Data using pandas
# df = pd.read_csv("Listings.csv")

# Load Airbnb listings dataset
df = pd.read_csv("Listings.csv", low_memory=False)

# Initial Exploration
df.head()  # Displays the first 5 rows

df.shape # Checks the size of the dataset

df.info() #Check column names, data types, and missing values

df.describe()# Generates summary stats for numeric columns

missing_values = df.isnull().sum().sort_values(ascending=False) # Tells the amount of missing/blank values each column has

# Tells you the percent amount of the columns missing cells
missing_percent = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   listing_id                   279712 non-null  int64  
 1   name                         279537 non-null  object 
 2   host_id                      279712 non-null  int64  
 3   host_since                   279547 non-null  object 
 4   host_location                278872 non-null  object 
 5   host_response_time           150930 non-null  object 
 6   host_response_rate           150930 non-null  float64
 7   host_acceptance_rate         166625 non-null  float64
 8   host_is_superhost            279547 non-null  object 
 9   host_total_listings_count    279547 non-null  float64
 10  host_has_profile_pic         279547 non-null  object 
 11  host_identity_verified       279547 non-null  object 
 12  neighbourhood                279712 non-null  object 
 13 

In [3]:
missing_percent

district                       86.767818
host_response_time             46.040928
host_response_rate             46.040928
host_acceptance_rate           40.429799
review_scores_value            32.814109
review_scores_location         32.810534
review_scores_checkin          32.809104
review_scores_accuracy         32.788368
review_scores_communication    32.779073
review_scores_cleanliness      32.771208
review_scores_rating           32.678255
bedrooms                       10.523324
host_location                   0.300309
name                            0.062564
host_identity_verified          0.058989
host_has_profile_pic            0.058989
host_is_superhost               0.058989
host_since                      0.058989
host_total_listings_count       0.058989
listing_id                      0.000000
minimum_nights                  0.000000
maximum_nights                  0.000000
longitude                       0.000000
price                           0.000000
amenities       

In [4]:
# df = df.drop(columns=["district"])
"district" in df.columns # can also do df.columns to show all the columns

True

In [5]:
# Putting all of the review columns into a list names review_cols
review_cols = [
    "review_scores_value",
    "review_scores_location",
    "review_scores_checkin",
    "review_scores_accuracy",
    "review_scores_communication",
    "review_scores_cleanliness",
    "review_scores_rating"
]

df[review_cols].head() # Looking to see if they were all added

,review_scores_value,review_scores_location,review_scores_checkin,review_scores_accuracy,review_scores_communication,review_scores_cleanliness,review_scores_rating
0,10.0,10.0,10.0,10.0,10.0,10.0,100.0
1,10.0,10.0,10.0,10.0,10.0,10.0,100.0
2,10.0,10.0,10.0,10.0,10.0,10.0,100.0
3,10.0,10.0,10.0,10.0,10.0,10.0,100.0
4,10.0,10.0,10.0,10.0,10.0,10.0,100.0


In [6]:
df[review_cols].isnull().sum() # Tells us how many null/na cells are in each column

review_scores_value            91785
review_scores_location         91775
review_scores_checkin          91771
review_scores_accuracy         91713
review_scores_communication    91687
review_scores_cleanliness      91665
review_scores_rating           91405
dtype: int64

In [7]:
df[review_cols] = df[review_cols].fillna(df[review_cols].median()) # Giving each medidan for each column and replacing any na values

In [8]:
df[review_cols].isnull().sum() # Re checking how many na cells there are after clearing them

review_scores_value            0
review_scores_location         0
review_scores_checkin          0
review_scores_accuracy         0
review_scores_communication    0
review_scores_cleanliness      0
review_scores_rating           0
dtype: int64

In [9]:
df[["host_response_time", "host_response_rate", "host_acceptance_rate"]].head(10) # shows the first 10 rows of each column
df[["host_response_time", "host_response_rate", "host_acceptance_rate"]].dtypes # Tells us the type each column is

host_response_time       object
host_response_rate      float64
host_acceptance_rate    float64
dtype: object

In [10]:
# Fill missing host response time values
# host_response_time is categorical, so missing values are treated as their own category.
# This avoids assuming that hosts with missing response data belong to the most common response-time group.

df["host_response_time"] = df["host_response_time"].replace("value", "Unknown")
df["host_response_time"] = df["host_response_time"].replace("unknown", "Unknown")
df["host_response_time"] = df["host_response_time"].fillna("Unknown")

# Check that missing values are handled and categories are clean
print(df["host_response_time"].isnull().sum())
df["host_response_time"].value_counts()

0


host_response_time
Unknown               128782
within an hour         83464
within a few hours     28891
within a day           23425
a few days or more     15150
Name: count, dtype: int64

In [11]:
host_rate_cols = ["host_response_rate", "host_acceptance_rate"] # Making a list called host_rate_cols

In [12]:
# Checking how many nulls we have in each column in the list
df[host_rate_cols].isnull().sum()

# Filling in all of the na's using fillna with the median of each column
df[host_rate_cols] = df[host_rate_cols].fillna(df[host_rate_cols].median())
df[host_rate_cols].isnull().sum() # Checking how many nulls there are left after filling them with na's

host_response_rate      0
host_acceptance_rate    0
dtype: int64

In [13]:
# Checking what other columns still have na's that we need to take care of
missing_values_after = df.isnull().sum().sort_values(ascending=False)
missing_values_after.head(15) # hows the first 15

district                       242700
bedrooms                        29435
host_location                     840
name                              175
host_is_superhost                 165
host_identity_verified            165
host_total_listings_count         165
host_has_profile_pic              165
host_since                        165
review_scores_checkin               0
review_scores_communication         0
amenities                           0
review_scores_location              0
review_scores_cleanliness           0
review_scores_accuracy              0
dtype: int64

In [14]:
# Making a new column called "bedrooms_missing" using "bedrooms" for model later
# Is checking each row and if missing then put 1 if not put 0 (astype(int) turns it from True and False into 1's and 0's)
df["bedrooms_missing"] = df["bedrooms"].isnull().astype(int)

# Tells us the amount of 0's and 1's
df["bedrooms_missing"].value_counts()

bedrooms_missing
0    250277
1     29435
Name: count, dtype: int64

In [15]:
# Filling all of the na's with the median
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].median())
# Checking is there are any nulls left
df["bedrooms"].isnull().sum()

np.int64(0)

In [16]:
# Reload the original host_since column because it was accidentally overwritten earlier
df_original = pd.read_csv("Listings.csv", low_memory=False)

df["host_since"] = df_original["host_since"]

# Convert host_since from text into a real datetime column
df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")

# Create host_years as a numeric feature
df["host_years"] = 2024 - df["host_since"].dt.year

# Check the result
df[["host_since", "host_years"]].head()

,host_since,host_years
0,2011-12-03,13.0
1,2013-11-29,11.0
2,2014-07-31,10.0
3,2013-12-17,11.0
4,2014-12-14,10.0


In [17]:
df[["host_since", "host_years"]].isnull().sum()

host_since    165
host_years    165
dtype: int64

In [18]:
df["host_years"] = df["host_years"].fillna(df["host_years"].median())

df["host_years"].isnull().sum()


np.int64(0)

In [19]:
host_cat_cols =[
    "host_is_superhost", "host_identity_verified"
]

In [20]:
df[host_cat_cols] = df[host_cat_cols].fillna("Unknown")

In [21]:
df[host_cat_cols].isnull().sum()

host_is_superhost         0
host_identity_verified    0
dtype: int64

In [22]:
# del host_cat_col
df["host_total_listings_count"] = df["host_total_listings_count"].fillna(df["host_total_listings_count"].median())
df["host_total_listings_count"].isnull().sum()
df = df.drop(columns=["district"])


In [23]:
missing_values_after = df.isnull().sum().sort_values(ascending=False)
missing_values_after.head(15)

host_location                  840
name                           175
host_since                     165
host_has_profile_pic           165
listing_id                       0
review_scores_cleanliness        0
price                            0
minimum_nights                   0
maximum_nights                   0
review_scores_rating             0
review_scores_accuracy           0
review_scores_checkin            0
bedrooms                         0
review_scores_communication      0
review_scores_location           0
dtype: int64

In [24]:
# Price outlier hanlding

# Checks the summary stats for the target variable
df["price"].describe()

# Check high-end price cutoffs
df["price"].quantile([0.90,0.95,0.99,0.995])

0.900     1200.0
0.950     2000.0
0.990     6500.0
0.995    10000.0
Name: price, dtype: float64

In [25]:
price_cutoff = df["price"].quantile(0.99)

price_cutoff

np.float64(6500.0)

In [26]:
df = df[(df["price"] > 0) & (df["price"] <= price_cutoff)]

In [27]:
df["price"].describe()

count    276874.000000
mean        433.198368
std         736.241305
min           1.000000
25%          75.000000
50%         150.000000
75%         450.000000
max        6500.000000
Name: price, dtype: float64

In [28]:
df.shape

(276874, 34)

In [29]:
# Define the target variable
target = "price"

# Define the input features used to predict price
feature_cols = [
    "city",
    "room_type",
    "property_type",
    "accommodates",
    "bedrooms",
    "bedrooms_missing",
    "minimum_nights",
    "maximum_nights",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "host_is_superhost",
    "host_identity_verified",
    "host_total_listings_count",
    "host_response_rate",
    "host_acceptance_rate",
    "host_years",
    "instant_bookable"
]

# Create X and y
X = df[feature_cols]
y = df[target]

In [30]:
print(X.shape)
print(y.shape)

X.isnull().sum().sort_values(ascending=False)

(276874, 18)
(276874,)


city                         0
room_type                    0
host_years                   0
host_acceptance_rate         0
host_response_rate           0
host_total_listings_count    0
host_identity_verified       0
host_is_superhost            0
review_scores_location       0
review_scores_cleanliness    0
review_scores_rating         0
maximum_nights               0
minimum_nights               0
bedrooms_missing             0
bedrooms                     0
accommodates                 0
property_type                0
instant_bookable             0
dtype: int64

In [31]:
# Encoding

# Identify categorical and numeric feature columns
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


Categorical features: ['city', 'room_type', 'property_type', 'host_is_superhost', 'host_identity_verified', 'instant_bookable']
Numeric features: ['accommodates', 'bedrooms', 'bedrooms_missing', 'minimum_nights', 'maximum_nights', 'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'host_total_listings_count', 'host_response_rate', 'host_acceptance_rate', 'host_years']


In [32]:
# Preprocessing

# Create preprocessing steps
# Categorical columns are one-hot encoded, while numeric columns are passed through unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [33]:
# Training

# Split the data into training and testing sets
# 80% of the data is used to train the model, and the other 20% is held out for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [34]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (221499, 18)
X_test: (55375, 18)
y_train: (221499,)
y_test: (55375,)


In [35]:
# Linear Regression

# Creating a Linear Regression pipeline
# The pipeline first preprcesses the data, then trains the model.
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [36]:
# Train the Linear Regression model
linear_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'room_type',
                                                   'property_type',
                                                   'host_is_superhost',
                                                   'host_identity_verified',
                                                   'instant_bookable']),
                                                 ('num', 'passthrough',
                                                  ['accommodates', 'bedrooms',
                                                   'bedrooms_missing',
                                                   'minimum_nights',
                                                   'maximum_nights',
                                                   'review_scores_rating',
                                                   'review_scores_cleanliness',
                                                   'review_scores_location',
                                                   'host_total_listings_count',
                                                   'host_response_rate',
                                                   'host_acceptance_rate',
                                                   'host_years'])])),
                ('model', LinearRegression())])

In [37]:
y_pred_linear = linear_model.predict(X_test)

In [38]:
# Evalute Linear Regression
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
linear_r2 = r2_score(y_test, y_pred_linear)

print("LinearRegression RMSE;", linear_rmse)
print("LinearRegression R^2:", linear_r2)

LinearRegression RMSE; 730.134832171748
LinearRegression R^2: -3.6867988180144806e-06


In [39]:
# Random Forest

# Creating a Random Forest pipeline
# The pipeline applies the same preprocessing steps, tehn trains a Random Forest model.

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [40]:
# Training it
rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city', 'room_type',
                                                   'property_type',
                                                   'host_is_superhost',
                                                   'host_identity_verified',
                                                   'instant_bookable']),
                                                 ('num', 'passthrough',
                                                  ['accommodates', 'bedrooms',
                                                   'bedrooms_missing',
                                                   'minimum_nights',
                                                   'maximum_nights',
                                                   'review_scores_rating',
                                                   'review_scores_cleanliness',
                                                   'review_scores_location',
                                                   'host_total_listings_count',
                                                   'host_response_rate',
                                                   'host_acceptance_rate',
                                                   'host_years'])])),
                ('model', RandomForestRegressor(n_jobs=-1, random_state=42))])

In [41]:
# Predictions
y_pred_rf = rf_model.predict(X_test)

In [42]:
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest R^2:", rf_r2)

Random Forest RMSE: 424.9202259075771
Random Forest R^2: 0.6613038395885518


In [43]:
model_results = pd.DataFrame({
    "Model": ["LinearRegression", "Random Forest"],
    "RMSE": [linear_rmse, rf_rmse],
    "R^2": [linear_r2, rf_r2]
})

model_results

,Model,RMSE,R^2
0,LinearRegression,730.134832,-0.000004
1,Random Forest,424.920226,0.661304


In [49]:
# Get the feature names created by one-hot encoding
encoded_cat_features = rf_model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .get_feature_names_out(categorical_features)

# Combine encoded categorical feature names with numeric feature names
all_feature_names = list(encoded_cat_features) + numeric_features

In [51]:
# Get feature importance values from the trained Random Forest model
importance = rf_model.named_steps["model"].feature_importances_

In [52]:
# Create a table of feature importances
feature_importance = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

,Feature,Importance
0,city_Bangkok,0.150429
163,bedrooms,0.142105
1,city_Cape Town,0.139536
162,accommodates,0.067257
4,city_Mexico City,0.057450
170,host_total_listings_count,0.057154
173,host_years,0.046756
165,minimum_nights,0.039885
166,maximum_nights,0.036377
172,host_acceptance_rate,0.035298


In [56]:
# Create a better grouping function for feature importance
# Categorical one-hot encoded features are grouped back to their original feature.
# Numeric features keep their original full column names.

def get_original_feature_name(feature):
    for cat_col in categorical_features:
        if feature.startswith(cat_col + "_"):
            return cat_col
    return feature

feature_importance_grouped = feature_importance.copy()

feature_importance_grouped["Original Feature"] = feature_importance_grouped["Feature"].apply(
    get_original_feature_name
)

grouped_importance = (
    feature_importance_grouped
    .groupby("Original Feature")["Importance"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

grouped_importance.head(15)

,Original Feature,Importance
0,city,0.387027
1,bedrooms,0.142105
2,accommodates,0.067257
3,property_type,0.067110
4,host_total_listings_count,0.057154
5,host_years,0.046756
6,minimum_nights,0.039885
7,maximum_nights,0.036377
8,host_acceptance_rate,0.035298
9,review_scores_rating,0.029436


In [57]:
# Select one sample listing from the test set
sample_listing = X_test.iloc[[0]]

# Predict the baseline price using the trained Random Forest model
base_price = rf_model.predict(sample_listing)[0]

base_price

np.float64(2258.84)

In [60]:
# Create pricing secnarios using teh predicted baseline price
scenarios = pd.DataFrame({
    "Scenario": [
        "20% Price Decrease",
        "10% Price Decrease",
        "Current Predicted Price",
        "10% Price Increase",
        "20% Price Increase"
    ],
    "Price Change": [-0.20, -0.10, 0.00, 0.10, 0.20],
    "Demand Multiplier": [1.20, 1.10, 1.00, 0.90, 0.80]
})

# Calculate adjusted price and estimated reveune
scenarios["Adjusted Price"] = base_price * (1 + scenarios["Price Change"])
scenarios["Estimated Revenue"] = scenarios["Adjusted Price"] * scenarios["Demand Multiplier"]

scenarios

,Scenario,Price Change,Demand Multiplier,Adjusted Price,Estimated Revenue
0,20% Price Decrease,-0.2,1.2,1807.072,2168.4864
1,10% Price Decrease,-0.1,1.1,2032.956,2236.2516
2,Current Predicted Price,0.0,1.0,2258.840,2258.8400
3,10% Price Increase,0.1,0.9,2484.724,2236.2516
4,20% Price Increase,0.2,0.8,2710.608,2168.4864


In [62]:
# Format pricing simulation results for readability
scenarios["Adjusted Price"] = scenarios["Adjusted Price"].round(2)
scenarios["Estimated Revenue"] = scenarios["Estimated Revenue"].round(2)

scenarios

,Scenario,Price Change,Demand Multiplier,Adjusted Price,Estimated Revenue
0,20% Price Decrease,-0.2,1.2,1807.07,2168.49
1,10% Price Decrease,-0.1,1.1,2032.96,2236.25
2,Current Predicted Price,0.0,1.0,2258.84,2258.84
3,10% Price Increase,0.1,0.9,2484.72,2236.25
4,20% Price Increase,0.2,0.8,2710.61,2168.49


In [64]:
# Predict price for a new Airbnb listing
# The new listing must include the same columns used to train the model.

new_listing = pd.DataFrame([{
    "city": "Bangkok",
    "room_type": "Entire home/apt",
    "property_type": "Apartment",
    "accommodates": 4,
    "bedrooms": 2,
    "bedrooms_missing": 0,
    "minimum_nights": 2,
    "maximum_nights": 30,
    "review_scores_rating": 95,
    "review_scores_cleanliness": 9,
    "review_scores_location": 9,
    "host_is_superhost": "t",
    "host_identity_verified": "t",
    "host_total_listings_count": 5,
    "host_response_rate": 95,
    "host_acceptance_rate": 90,
    "host_years": 6,
    "instant_bookable": "t"
}])

predicted_price = float(rf_model.predict(new_listing)[0])

print(f"Predicted Airbnb Price: ${predicted_price:,.2f}")

Predicted Airbnb Price: $1,804.82


In [65]:
import joblib

#Saves the trained Random Forst Pipeline
joblib.dump(rf_model, "random_forest_airbnb_model.pkl")

['random_forest_airbnb_model.pkl']

In [66]:
loaded_rf_model = joblib.load("random_forest_airbnb_model.pkl")

loaded_predicted_price = float(loaded_rf_model.predict(new_listing)[0])

print(f"Loaded Model Predicted Airbnb Price: ${loaded_predicted_price:,.2f}")

Loaded Model Predicted Airbnb Price: $1,804.82


In [ ]:
'''Project Summary

This project built an Airbnb listing price prediction model using property, host, location, and review features. I cleaned the dataset by handling missing values based on feature type, removing the `district` column due to high missingness, filling numeric columns with median values, and creating an `Unknown` category for missing categorical values.

I engineered new features such as `host_years` to represent host experience and `bedrooms_missing` to preserve whether bedroom count was originally missing. I also removed extreme price outliers above the 99th percentile to focus the model on typical Airbnb listings.

I trained two models: Linear Regression and Random Forest. Linear Regression performed poorly as a baseline, with an R² near zero. Random Forest performed much better, reducing RMSE to about $424.92 and improving R² to about 0.66.

Feature importance showed that city, bedrooms, accommodates, property type, and host-related variables were among the strongest predictors. These results show predictive usefulness, not causation.

I also added a new-listing prediction example, where a user can enter listing information and the trained model estimates the Airbnb price. Finally, I created a simple pricing scenario simulation using the model’s predicted price as a baseline to compare estimated revenue under different price changes.

Limitations include the lack of actual occupancy or booking-demand data, possible remaining variation from unobserved factors, and the need for further model tuning. Future improvements could include adding neighborhood, latitude/longitude, seasonality, log-transformed price, hyperparameter tuning, and real booking data.
'''